## Libraries set-up

In [ ]:
import yfinance as yf
import pandas as pd
import os
from datetime import datetime, timedelta

import plotly.graph_objects as go
import plotly.express as px

## A) Dataset build-up

In [ ]:
# 1. Configuration: TICKERS / Sector mapping
sector_map = {
    'Solar': ['FSLR', 'ENPH', 'SEDG', 'NXT', 'CSIQ', 'RUN', 'SHLS', 'DQ', 'JKS', 'SPWR', 'MAXN'],
    'Wind': ['GEV', 'VWS.CO', 'BEP', 'CWEN', 'ORA', 'TPIC', 'ORSTED.CO'],
    'Nuclear': ['CEG', 'CCJ', 'UUUU', 'LEU', 'UEC', 'SMR', 'OKLO', 'BWXT', 'FLR', 'NXE', 'DNN','IMSR'],
    'Hydrogen': ['BE', 'PLUG', 'NEL.OL', 'ITM.L', 'HYSR', 'FCEL', 'BLDP'],
    'Storage & Grid': ['TSLA', 'FLNC', 'STEM', 'EOSE', 'GWH', 'NRGV', 'GNRC', 'AYI', 'HUBB', 'PWR', 'VRT', 'EAT'],
    'Integrated Energy': ['NEE', 'XOM', 'CVX', 'SHEL', 'TTE', 'BP', 'EQNR', 'IBE.MC', 'ENEL.MI']
}

all_tickers = [ticker for sublist in sector_map.values() for ticker in sublist]
ticker_to_sector = {ticker: sector for sector, tickers in sector_map.items() for ticker in tickers}

# 2. Fetch Fresh Data (Summary and Flipped Historical)
def fetch_data():
    summary_data, historical_prices = [], pd.DataFrame()
    end_date = datetime.now()
    start_date = end_date - timedelta(days=2*365)

    print(f"Fetching data for {len(all_tickers)} companies...")
    for ticker in all_tickers:
        try:
            stock = yf.Ticker(ticker)
            info, hist = stock.info, stock.history(start=start_date, end=end_date)['Close']
            if hist.empty: continue
            hist.index = hist.index.tz_localize(None).date

            summary_data.append({
                'Ticker': ticker, 'Name': info.get('longName', ticker), 'Sector': ticker_to_sector.get(ticker),
                'Market Capitalization': info.get('marketCap'), 'Trading Volume (Avg)': info.get('averageVolume'),
                '52 Week High': info.get('fiftyTwoWeekHigh'), '52 Week Low': info.get('fiftyTwoWeekLow'),
                'Last Price': info.get('currentPrice') or info.get('regularMarketPreviousClose')
            })
            historical_prices[ticker] = hist
        except: continue
    return pd.DataFrame(summary_data), historical_prices

df_summary, df_history_wide = fetch_data()




Fetching data for 58 companies...


In [ ]:
# 3. FLIP THE AXIS & ENRICH DATA
# Melt Wide to Long
df_history_long = df_history_wide.reset_index().melt(id_vars='index', var_name='Ticker', value_name='Price')
df_history_long.rename(columns={'index': 'Date'}, inplace=True)

# Calculate Price Change regarding previous close (per Ticker)
df_history_long.dropna(subset=['Price'], inplace=True)
df_history_long = df_history_long.sort_values(['Ticker', 'Date'])
df_history_long['Price Change'] = df_history_long.groupby('Ticker')['Price'].pct_change(fill_method=None)

# ADDED: Calculate Cumulative Change (Return since the start of the 2-year period)
df_history_long['Cumulative Change'] = df_history_long.groupby('Ticker')['Price'].transform(lambda x: (x / x.iloc[0]) - 1)

# Sort summary to identify Top 5
df_summary = df_summary.sort_values(by='Market Capitalization', ascending=False)
df_top_5 = df_summary.head(5)
top_5_tickers = df_top_5['Ticker'].tolist()

# NEW: Merge Name and Create 'Top Five?' column
df = df_history_long.merge(df_summary[['Ticker', 'Name']], on='Ticker', how='left')
df['Top Five?'] = df['Ticker'].apply(lambda x: 'Yes' if x in top_5_tickers else 'No')

## B) Plotting Cumulative Change

In [ ]:
#1 Timeframe set-up
min_date, max_date = df['Date'].min(), df['Date'].max()
first_month_label = min_date.strftime("%b %Y")
last_month_label = max_date.strftime("%b %Y")

# 2. Setup Figure and Global Variables
fig = go.Figure()
tickers = sorted(df['Ticker'].unique())
colors = px.colors.qualitative.Plotly
all_annotations = []
trace_metadata = []

# 3. Build Traces and Badge Annotations
for i, ticker in enumerate(tickers):
    df_ticker = df[df['Ticker'] == ticker]
    is_top_five = df_ticker['Top Five?'].iloc[0]
    stock_name = df_ticker['Name'].iloc[0] # Full Company Name for the Slicer
    color = colors[i % len(colors)]

    valid_data = df_ticker.dropna(subset=['Cumulative Change'])
    if valid_data.empty: continue
    last_valid_row = valid_data.iloc[-1]

    # Line Trace - Legend shows Ticker, Hover shows Full Name
    fig.add_trace(go.Scatter(
        x=df_ticker['Date'], y=df_ticker['Cumulative Change'],
        mode='lines',
        name=ticker, # <--- Keeps TICKER in the legend/labels
        line=dict(color=color, width=2),
        meta={'name': stock_name, 'is_top_five': is_top_five}, # <--- Stores FULL NAME for slicer
        visible=True,
        hovertemplate=f"<b>{stock_name}</b><br>Change: %{{y:.2%}}<extra></extra>"
    ))

    # Marker Anchor
    fig.add_trace(go.Scatter(
        x=[last_valid_row['Date']], y=[last_valid_row['Cumulative Change']],
        mode='markers', marker=dict(color=color, size=4),
        showlegend=False, visible=True, hoverinfo='skip'
    ))

    # Metadata for Slicers (Keyed by Full Name)
    trace_metadata.extend([{'name': stock_name, 'is_top_five': is_top_five}] * 2)

    # Badge Annotation (Displays Ticker + Percentage)
    all_annotations.append(dict(
        x=last_valid_row['Date'], y=last_valid_row['Cumulative Change'],
        text=f"<b>{last_valid_row['Cumulative Change']:,.2%}</b>",
        showarrow=False, xanchor="left", xshift=10,
        font=dict(color="white", size=10, family="Arial Black"),
        bgcolor=color, bordercolor=color, borderwidth=2, borderpad=4,
        visible=True
    ))

# 4. Slicer Logic - Filtering by Company Name
def get_args(name_filter=None, top_5_only=False):
    trace_vis, ann_vis = [], []
    for i, meta in enumerate(trace_metadata):
        name, top5 = meta['name'], meta['is_top_five']
        is_visible = True
        if name_filter and name != name_filter: is_visible = False
        if top_5_only and top5 != 'Yes': is_visible = False
        trace_vis.append(is_visible)
        if i % 2 == 0: ann_vis.append(is_visible)

    updated_anns = [dict(ann, visible=vis) for ann, vis in zip(all_annotations, ann_vis)]
    return [{"visible": trace_vis}, {"annotations": updated_anns}]

# 5. Build Dropdown Options using Company Names
available_names = sorted(list(set([m['name'] for m in trace_metadata])))
name_dropdown = [dict(label="All Companies", method="update", args=get_args())]

for name in available_names:
    name_dropdown.append(dict(label=name, method="update", args=get_args(name_filter=name)))

# 6. Layout and Formatting
fig.update_layout(
    updatemenus=[
        dict(type="buttons", direction="left", buttons=[
                dict(label="Show All", method="update", args=get_args()),
                dict(label="Top 5 Only", method="update", args=get_args(top_5_only=True))
            ], x=0.0, y=1.22, xanchor="left", yanchor="top"),
        dict(type="dropdown", direction="down", buttons=name_dropdown, x=0.25, y=1.22, xanchor="left", yanchor="top")
    ],
    xaxis=dict(
        title="Date",
        range=[min_date, max_date],
        type='date',
        tickformat="%b %Y", # e.g. Mar 2024
        dtick="M1",         # <--- Forces EVERY month to show
        tickangle=-45       # Tilts labels to avoid overlap
    ),
    yaxis=dict(title="Cumulative Change", tickformat=".0%"),
    template="plotly_white", height=750,
    margin=dict(r=300, t=100),
    legend=dict(orientation="v", x=1.05, y=1, xanchor="left"),
    hovermode="x unified",
    annotations=all_annotations
)

fig.show()